# SIH 2026 - Problem Statement SIH26168
# AI-ML Based Intelligent Dead Reckoning for Seamless Navigation
## Notebook 01: Dataset Discovery, Schema Verification & Signal Analysis

### Overview
Smartphone-based Dead Reckoning requires continuous inertial sensor signals (accelerometer, gyroscope) matched against vehicle ground-truth telemetry. This notebook inspects the raw dataset, maps sensor channels, validates sampling frequency, and profiles usable driving sequences without making assumptions about folder layouts.

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Add project root and src to import path
PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from common_utils import (
    DATASET_ROOT, discover_sync_pairs, load_s_file, load_v_file,
    compute_sampling_rate, ARTIFACTS_DIR
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATASET_ROOT}")

Project root: /home/sibsankar_de/projects/sih_2026/ml_models
Dataset root: /home/sibsankar_de/projects/sih_2026/ml_models/datasets


### 1. Dataset Discovery
Recursively scan the dataset directory to locate all smartphone sensor CSV files (`S-*.csv`) and vehicle CAN-bus telemetry files (`V-*.csv`), and pair them by sequence identifier.

In [2]:
all_csvs = list(DATASET_ROOT.rglob("*.csv"))
s_files = [f for f in all_csvs if f.name.startswith("S-")]
v_files = [f for f in all_csvs if f.name.startswith("V-")]
sync_pairs = discover_sync_pairs()

print(f"Total CSV files found:         {len(all_csvs)}")
print(f"Smartphone sensor files (S-*): {len(s_files)}")
print(f"Vehicle CAN-bus files (V-*):   {len(v_files)}")
print(f"Synchronized S-V pairs found:  {len(sync_pairs)}")

Total CSV files found:         564
Smartphone sensor files (S-*): 241
Vehicle CAN-bus files (V-*):   323
Synchronized S-V pairs found:  72


### 2. Sensor Schema & Channel Verification
Examine the exact columns recorded in the smartphone IMU files versus the vehicle CAN-bus telemetry files.

In [3]:
sample_pair = sync_pairs[0]
s_sample = load_s_file(sample_pair['s_file'])
v_sample = load_v_file(sample_pair['v_file'])

print(f"Sample sequence: {sample_pair['seq_id']} (Driver {sample_pair['driver']})")
print(f"Smartphone columns ({len(s_sample.columns)}):\n{list(s_sample.columns[:12])} ...")
print(f"\nVehicle CAN columns ({len(v_sample.columns)}):\n{list(v_sample.columns[:12])} ...")

Sample sequence: M (Driver B)
Smartphone columns (24):
['gps_lat', 'gps_lon', 'gps_alt', 'gps_speed_kmh', 'gps_accuracy', 'gps_orientation', 'gps_satellites', 'time_since_start_ms', 'date', 'acc_x', 'acc_y', 'acc_z'] ...

Vehicle CAN columns (29):
['gps_satellites', 'time_since_day_start_s', 'lat', 'lon', 'velocity_kmh', 'heading', 'height_km', 'vertical_velocity_kmh', 'sample_period_s', 'steering_angle', 'wheel_speed_fl', 'wheel_speed_fr'] ...


### 3. Sequence Profiling & Sampling Rate Analysis
Verify the temporal alignment, sampling frequency ($Hz$), driving duration, and velocity distribution across all sequences.

In [4]:
stats_path = ARTIFACTS_DIR / "configs" / "sequence_statistics.csv"
if stats_path.exists():
    stats_df = pd.read_csv(stats_path)
    print(f"Inspected {len(stats_df)} sequences.")
    print(f"Sampling rate: {stats_df['s_sampling_rate_hz'].median():.1f} Hz (Smartphone), {stats_df['v_sampling_rate_hz'].median():.1f} Hz (Vehicle)")
    print(f"Total driving duration: {stats_df['duration_seconds'].sum()/3600:.2f} hours")
    print(f"Average vehicle speed:  {stats_df['speed_mean_kmh'].mean():.1f} km/h (Max: {stats_df['speed_max_kmh'].max():.1f} km/h)")
    display(stats_df[['seq_id', 'driver', 's_rows', 's_sampling_rate_hz', 'duration_seconds', 'speed_mean_kmh']].head(10))

Inspected 72 sequences.
Sampling rate: 10.0 Hz (Smartphone), 10.0 Hz (Vehicle)


KeyError: 'duration_seconds'

### 4. Summary & Saved Artifacts
Dataset inspection outputs are cached in `artifacts/configs/` for use in the model training notebooks.

In [ ]:
summary_path = ARTIFACTS_DIR / "configs" / "dataset_summary.json"
with open(summary_path) as f:
    summary = json.load(f)

for k, v in summary.items():
    print(f"  {k:25s}: {v}")